In [1]:
import zipfile
import os

zip_path = r'/content/old_data (do not use)-20261124T012928Z-1-001.zip'
extract_path = '/content/CS117_Project_Images'

# Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Successfully unzipped to: {extract_path}")

# List the contents of the extracted directory
print("\nExtracted contents:")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 4 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

Successfully unzipped to: /content/CS117_Project_Images

Extracted contents:
CS117_Project_Images/
    old_data (do not use)/
        Medium Distinctive/
            PXL_20230227_032738002.jpg
            PXL_20230221_122426759.jpg
            PXL_20260908_021115988.jpg
            PXL_20250916_182210441.jpg
            PXL_20260722_071229584.jpg
            PXL_20231119_003155535.jpg
            PXL_20250801_114821486.jpg
            IMG_3390.jpeg
            PXL_20260703_065147127.jpg
            PXL_20230710_010318943.jpg
            PXL_20261011_013512014.jpg
            PXL_20250121_012824517.jpg
            IMG_1577.jpeg
            PXL_20230221_141011778.jpg
            PXL_20260610_165434353.jpg
            PXL_20240504_110218409.jpg
            PXL_20260322_024831940.jpg
            PXL_20260830_160946901.jpg
            PXL_20261113_140442528.jpg
            PXL_20230907_160558848.jpg
            PXL_20261024_010713379.jpg
            PXL_20260221_221917000.jpg
            PX

In [2]:
!pip install geoclip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 24.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 104.8 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geoclip]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [3]:
from pathlib import Path

directory = Path("/content/CS117_Project_Images")

# Get a list of Path objects
file_paths = [p for p in directory.rglob("*") if p.is_file()]
print(f"{len(file_paths)} files")

375 files


In [4]:
from geoclip import GeoCLIP
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GeoCLIP(from_pretrained=True)
model.to(device)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

GeoCLIP(
  (image_encoder): ImageEncoder(
    (CLIP): CLIPModel(
      (text_model): CLIPTextModel(
        (embeddings): CLIPTextEmbeddings(
          (token_embedding): Embedding(49408, 768)
          (position_embedding): Embedding(77, 768)
        )
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-11): 12 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=768, out_features=768, bias=True)
                (v_proj): Linear(in_features=768, out_features=768, bias=True)
                (q_proj): Linear(in_features=768, out_features=768, bias=True)
                (out_proj): Linear(in_features=768, out_features=768, bias=True)
              )
              (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
              (mlp): CLIPMLP(
                (activation_fn): QuickGELUActivation()
                (fc1): Linear(in_features=768, out_features=3072, bias=True)
       

In [5]:
top_pred_gps, top_pred_prob = model.predict(file_paths[0], top_k=5)

print("Top 5 GPS Predictions")
print("=====================")
for i in range(5):
    lat, lon = top_pred_gps[i]
    print(f"Prediction {i+1}: ({lat:.6f}, {lon:.6f})")
    print(f"Probability: {top_pred_prob[i]:.6f}")
    print("")

Top 5 GPS Predictions
Prediction 1: (36.985004, 68.774414)
Probability: 0.046457

Prediction 2: (38.349998, 43.000000)
Probability: 0.040768

Prediction 3: (38.503395, 43.398125)
Probability: 0.037295

Prediction 4: (34.936989, 69.260651)
Probability: 0.030054

Prediction 5: (34.936989, 69.260651)
Probability: 0.030054



In [6]:
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

def get_gps_decimal(image_path):
    def convert_to_degrees(value):
        d = float(value[0])
        m = float(value[1])
        s = float(value[2])
        return d + (m / 60.0) + (s / 3600.0)

    try:
        img = Image.open(image_path)
        exif_data = img._getexif()
        if not exif_data:
            print("No EXIF metadata found.")
            return None

        gps_info = {}
        for tag, value in exif_data.items():
            decoded = TAGS.get(tag, tag)
            if decoded == "GPSInfo":
                for t in value:
                    sub_decoded = GPSTAGS.get(t, t)
                    gps_info[sub_decoded] = value[t]

        if "GPSLatitude" in gps_info and "GPSLongitude" in gps_info:
            lat = convert_to_degrees(gps_info["GPSLatitude"])
            if gps_info.get("GPSLatitudeRef", "N") != "N":
                lat = -lat

            lon = convert_to_degrees(gps_info["GPSLongitude"])
            if gps_info.get("GPSLongitudeRef", "E") != "E":
                lon = -lon

            # print(f"Latitude (Decimal): {lat:.6f}")
            # print(f"Longitude (Decimal): {lon:.6f}")
            return lat, lon
        else:
            print("No GPS metadata found in EXIF.")
            return None
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

# Example usage:
# get_gps_decimal(file_paths[0])

In [7]:
import json
import torch

def get_all_locations_top5_json(paths_list):
    results = []

    # Ensure model is on the correct device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for p in paths_list:
        path_str = str(p)

        # 1. Get actual GPS coordinates from metadata
        actual_gps = get_gps_decimal(path_str)
        actual_lat, actual_lon = actual_gps if actual_gps else (None, None)

        # 2. Get top 5 predictions using GeoCLIP
        predictions_list = []
        try:
            top_pred_gps, top_pred_prob = model.predict(path_str, top_k=5)
            for i in range(len(top_pred_gps)):
                lat, lon = float(top_pred_gps[i][0]), float(top_pred_gps[i][1])
                prob = float(top_pred_prob[i])
                predictions_list.append({
                    "latitude": lat,
                    "longitude": lon,
                    "probability": prob
                })
        except Exception as e:
            pass

        results.append({
            "file_path": path_str,
            "actual_gps": {
                "latitude": actual_lat,
                "longitude": actual_lon
            },
            "top_5_predictions": predictions_list
        })

    return results

# To run and inspect on a subset:
# print(get_all_locations_top5_json(file_paths[:3]))

In [8]:
result = get_all_locations_top5_json(file_paths)

No GPS metadata found in EXIF.
No GPS metadata found in EXIF.
No GPS metadata found in EXIF.
No GPS metadata found in EXIF.


In [10]:
invalid_data = [x for x in result if x["actual_gps"]["latitude"] is None]
invalid_count = len(invalid_data)
print(f"{invalid_count} images had no gps data")

4 images had no gps data


In [13]:
valid_data = [x for x in result if x["actual_gps"]["latitude"] is not None]
valid_count = len(valid_data)
print(f"{valid_count} images had gps data")

371 images had gps data


In [14]:
with open(r"/content/cs117_images_gps.json", "w") as f:
  json.dump(valid_data, f, indent=4)
  print("Done!")

Done!


In [15]:
import math

def check_predictions_within_radius(data_list, max_distance_miles=5.0):
    def haversine_distance(coord1, coord2):
        # Coordinates in decimal degrees
        lat1, lon1 = coord1
        lat2, lon2 = coord2

        if None in (lat1, lon1, lat2, lon2):
            return float('inf')

        # Mean radius of Earth in miles
        R = 3958.8

        phi1 = math.radians(lat1)
        phi2 = math.radians(lat2)
        delta_phi = math.radians(lat2 - lat1)
        delta_lambda = math.radians(lon2 - lon1)

        a = math.sin(delta_phi / 2.0) ** 2 + \
            math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2.0) ** 2

        c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
        return R * c

    accuracy_dict = {}

    for item in data_list:
        path = item.get("file_path")
        actual = item.get("actual_gps", {})
        actual_coords = (actual.get("latitude"), actual.get("longitude"))

        is_within_limit = False

        # Check if actual GPS exists
        if actual_coords[0] is not None and actual_coords[1] is not None:
            for pred in item.get("top_5_predictions", []):
                pred_coords = (pred.get("latitude"), pred.get("longitude"))
                distance = haversine_distance(actual_coords, pred_coords)
                if distance <= max_distance_miles:
                    is_within_limit = True
                    break

        accuracy_dict[path] = is_within_limit

    return accuracy_dict

# Example usage with the existing 'output' variable:
# evaluation_results = check_predictions_within_radius(output)
# print(list(evaluation_results.items())[:5])

In [16]:
acc_dict = check_predictions_within_radius(valid_data)

In [17]:
accuracy = sum(acc_dict.values())/len(acc_dict.values())
print(f"Accuracy: {accuracy}")

Accuracy: 0.5471698113207547
